# Maia Lite 0.3.0 — Pretraining Engine v3.8 (GPU-budget mode)

Resume existing checkpoint and continue to global target **309,616 updates**. No routine validation or corpus/optimizer audits. Periodic checkpoints at global steps divisible by **10,000**; additionally save at controlled session end, global completion, and catchable exceptions. Keep lightweight training logs.

**Cost/risk:** unexpected Colab disconnects can lose up to roughly 10,000 updates; checkpoint I/O and session-end saves still take time. No notebook can guarantee saving if the runtime is forcibly terminated. The first periodic checkpoint after step 17,200 is step 20,000. This notebook has been statically checked but not executed on an A100.

**Data policy:** `experimental_unreviewed` is not a rights approval.


In [ ]:
from pathlib import Path
import os, json, math, time, random, hashlib, bisect, gc, datetime, shutil
from dataclasses import asdict
from collections import defaultdict
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').is_dir(): drive.mount('/content/drive')
except ImportError:
    print('Colab Drive not available; local validation only')
PROJECT=Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat')
SSD=Path('/content/maia_v8_ssd/dataset')
MANIFEST=SSD/'dataset_manifest.json'
APPROVALS=PROJECT/'corpus_v8/approvals/shard_eligibility_v1.jsonl'
PRODUCTION=False  # Never switch on until independent rights review and preflight pass
SEED=20261105
print('Torch:',torch.__version__,'CUDA:',torch.cuda.is_available())
print('SSD manifest present:',MANIFEST.is_file())

Mounted at /content/drive
Torch: 2.14.1+cu130 CUDA: True
SSD manifest present: False


## 1 — Prepare / reuse SSD corpus

In [ ]:
# CELL 3 — AUTO-BOOTSTRAP: run once after configuration, before corpus integrity gate.
import sqlite3, zipfile
from collections import defaultdict

CORPUS_ROOT = PROJECT / 'corpus_v8/runs/acquisition_v8_5_001/pretraining_production_v1_full'
PACK_DIR = CORPUS_ROOT / 'packed_zip_v1'
DB_PATH = CORPUS_ROOT / 'recovery_audit_v4/state_operational_v5.sqlite3'
SSD_PARENT = SSD.parent
WORK = SSD_PARENT / 'dataset.partial'
ZIP_TMP = SSD_PARENT / 'current_pack.zip'
BLOCK_BYTES = 2048 * 4
PACK_LIMIT = 1024**3

def _hash_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(4 * 1024**2), b''):
            h.update(b)
    return h.hexdigest()

def _check_local_dataset():
    if not MANIFEST.is_file():
        return False
    try:
        m = json.loads(MANIFEST.read_text())
        if not (m.get('status') == 'complete' and m.get('total_blocks') == 3607119
                and m.get('total_shards') == 60612 and m.get('tokens_per_block') == 2048
                and m.get('dtype') == '<u4' and len(m.get('splits', [])) == 18):
            return False
        for e in m['splits']:
            f = SSD / e['relative_path']
            idx = SSD / e['index_path']
            if not f.is_file() or f.stat().st_size != e['blocks'] * BLOCK_BYTES or not idx.is_file():
                return False
        return True
    except (OSError, ValueError, KeyError):
        return False

def _build_local_dataset():
    if _check_local_dataset():
        print('REUSE: SSD dataset already complete:', SSD)
        return
    if SSD.exists():
        raise RuntimeError('Existing SSD dataset is incomplete; inspect/remove it manually: ' + str(SSD))
    assert PACK_DIR.is_dir(), f'ZIP pack directory missing: {PACK_DIR}'
    assert DB_PATH.is_file(), f'SQLite catalog missing: {DB_PATH}'
    SSD_PARENT.mkdir(parents=True, exist_ok=True)
    conn = sqlite3.connect(DB_PATH.as_uri() + '?mode=ro', uri=True, timeout=60)
    try:
        rows = conn.execute('SELECT source, split, shard_index, path, blocks, sha256 FROM shard ORDER BY source, split, shard_index').fetchall()
    finally:
        conn.close()
    assert len(rows) == 60612 and sum(r[4] for r in rows) == 3607119
    packs, group, size = [], [], 0
    for r in rows:
        nbytes = r[4] * BLOCK_BYTES
        if group and size + nbytes > PACK_LIMIT:
            packs.append(group)
            group, size = [], 0
        group.append(r)
        size += nbytes
    if group:
        packs.append(group)
    assert len(packs) == 28
    required = sum(r[4] for r in rows) * BLOCK_BYTES
    free = shutil.disk_usage(SSD_PARENT).free
    print(f'Corpus: {required/1024**3:.2f} GiB | free: {free/1024**3:.2f} GiB')
    assert free >= required + 2*1024**3, 'Insufficient SSD space'
    if WORK.exists():
        shutil.rmtree(WORK)  # interrupted build only; never touches completed dataset
    WORK.mkdir(parents=True)
    handles, indices = {}, {}
    counts, shards = defaultdict(int), defaultdict(int)
    start = time.monotonic()
    try:
        for i, pack in enumerate(packs):
            src, split = pack[0][:2]
            name = f'pack_{i:04d}_{src}_{split}.zip'
            meta = json.loads((PACK_DIR / (name + '.json')).read_text())
            assert meta['status'] == 'complete' and meta['zip_name'] == name and meta['count'] == len(pack)
            expected = {f'{r[0]}/{r[1]}/{r[0]}_{r[1]}_{r[2]:06d}.bin':r for r in pack}
            ZIP_TMP.unlink(missing_ok=True)
            print(f'[{i+1}/28] {name}', flush=True)
            shutil.copyfile(PACK_DIR / name, ZIP_TMP)
            assert ZIP_TMP.stat().st_size == meta['zip_bytes']
            assert _hash_file(ZIP_TMP) == meta['zip_sha256']
            with zipfile.ZipFile(ZIP_TMP) as z:
                assert len(z.infolist()) == len(pack)
                seen = set()
                for member in z.infolist():
                    arc = member.filename
                    assert arc in expected and arc not in seen
                    assert member.compress_type == zipfile.ZIP_STORED
                    seen.add(arc)
                    source, split, shard_idx, _, blocks, sha = expected[arc]
                    assert member.file_size == blocks * BLOCK_BYTES
                    key = (source, split)
                    folder = WORK / source
                    folder.mkdir(exist_ok=True)
                    if key not in handles:
                        handles[key] = open(folder / f'{split}.bin', 'wb')
                        indices[key] = open(folder / f'{split}.index.jsonl', 'w', encoding='utf-8')
                    digest = hashlib.sha256()
                    written = 0
                    with z.open(member) as f:
                        while True:
                            buf = f.read(4 * 1024**2)
                            if not buf:
                                break
                            handles[key].write(buf)
                            digest.update(buf)
                            written += len(buf)
                    assert written == blocks * BLOCK_BYTES and digest.hexdigest() == sha.lower(), f'Shard checksum mismatch: {arc}'
                    indices[key].write(json.dumps({'shard_index': shard_idx, 'start_block': counts[key], 'blocks': blocks, 'sha256': sha}) + '\n')
                    counts[key] += blocks
                    shards[key] += 1
                assert seen == set(expected)
            ZIP_TMP.unlink()
            print(f'  OK | {i+1}/28 | {(time.monotonic()-start)/60:.1f} min', flush=True)
        for f in [*handles.values(), *indices.values()]:
            f.flush()
            os.fsync(f.fileno())
            f.close()
        handles.clear(); indices.clear()
        assert sum(counts.values()) == 3607119 and sum(shards.values()) == 60612
        manifest = {'status': 'complete', 'tokens_per_block': 2048, 'dtype': '<u4',
                    'total_blocks': 3607119, 'total_shards': 60612, 'splits': []}
        for (source, split), blocks in sorted(counts.items()):
            f = WORK / source / f'{split}.bin'
            assert f.stat().st_size == blocks * BLOCK_BYTES
            manifest['splits'].append({'source': source, 'split': split, 'blocks': blocks,
              'shards': shards[(source,split)], 'relative_path': f'{source}/{split}.bin',
              'index_path': f'{source}/{split}.index.jsonl'})
        assert len(manifest['splits']) == 18
        (WORK / 'dataset_manifest.json').write_text(json.dumps(manifest, indent=2))
        WORK.rename(SSD)
        print('PASS: SSD DATASET PREPARED', flush=True)
    finally:
        for f in [*handles.values(), *indices.values()]:
            if not f.closed:
                f.close()
        ZIP_TMP.unlink(missing_ok=True)

_build_local_dataset()
assert _check_local_dataset(), 'SSD-first bootstrap failed validation'
print('PASS: SSD-FIRST DATASET READY')


Corpus: 27.52 GiB | free: 235.28 GiB
[1/28] pack_0000_fineweb2_es_test.zip
  OK | 1/28 | 0.5 min
[2/28] pack_0001_fineweb2_es_train.zip
  OK | 2/28 | 1.0 min
[3/28] pack_0002_fineweb2_es_train.zip
  OK | 3/28 | 1.6 min
[4/28] pack_0003_fineweb2_es_train.zip
  OK | 4/28 | 2.1 min
[5/28] pack_0004_fineweb2_es_train.zip
  OK | 5/28 | 2.7 min
[6/28] pack_0005_fineweb2_es_train.zip
  OK | 6/28 | 3.3 min
[7/28] pack_0006_fineweb2_pt_train.zip
  OK | 7/28 | 3.8 min
[8/28] pack_0007_fineweb2_pt_train.zip
  OK | 8/28 | 4.4 min
[9/28] pack_0008_fineweb2_pt_train.zip
  OK | 9/28 | 4.9 min
[10/28] pack_0009_fineweb2_pt_train.zip
  OK | 10/28 | 5.4 min
[11/28] pack_0010_fineweb2_pt_train.zip
  OK | 11/28 | 5.9 min
[12/28] pack_0011_fineweb2_pt_train.zip
  OK | 12/28 | 6.4 min
[13/28] pack_0012_fineweb2_pt_train.zip
  OK | 13/28 | 6.9 min
[14/28] pack_0013_fineweb2_pt_train.zip
  OK | 14/28 | 7.4 min
[15/28] pack_0014_fineweb2_pt_train.zip
  OK | 15/28 | 7.9 min
[16/28] pack_0015_fineweb_edu_en_trai

## 2 — Original model architecture

In [ ]:
from dataclasses import dataclass
@dataclass
class Config:
    vocab_size:int=50257
    block_size:int=2048
    n_layer:int=36
    n_head:int=16
    n_embd:int=1024
    dropout:float=0.0
    bias:bool=True
cfg=Config()
class Block(nn.Module):
    def __init__(self,c):
        super().__init__()
        d=c.n_embd
        self.ln1=nn.LayerNorm(d)
        self.qkv=nn.Linear(d,3*d,bias=c.bias)
        self.proj=nn.Linear(d,d,bias=c.bias)
        self.ln2=nn.LayerNorm(d)
        self.fc=nn.Linear(d,4*d,bias=c.bias)
        self.out=nn.Linear(4*d,d,bias=c.bias)
        self.nh=c.n_head
    def forward(self,x):
        b,t,d=x.shape
        q,k,v=self.qkv(self.ln1(x)).chunk(3,dim=-1)
        q=q.view(b,t,self.nh,d//self.nh).transpose(1,2)
        k=k.view(b,t,self.nh,d//self.nh).transpose(1,2)
        v=v.view(b,t,self.nh,d//self.nh).transpose(1,2)
        a=F.scaled_dot_product_attention(q,k,v,is_causal=True,dropout_p=0.0)
        x=x+self.proj(a.transpose(1,2).contiguous().view(b,t,d))
        x=x+self.out(F.gelu(self.fc(self.ln2(x)),approximate='tanh'))
        return x
class MaiaGPT(nn.Module):
    def __init__(self,c):
        super().__init__();self.cfg=c
        self.tok=nn.Embedding(c.vocab_size,c.n_embd)
        self.pos=nn.Embedding(c.block_size,c.n_embd)
        self.blocks=nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        self.norm=nn.LayerNorm(c.n_embd)
        self.head=nn.Linear(c.n_embd,c.vocab_size,bias=False)
        self.head.weight=self.tok.weight
        self.apply(self.init_weights)
    @staticmethod
    def init_weights(m):
        if isinstance(m,(nn.Linear,nn.Embedding)):
            nn.init.normal_(m.weight,mean=0,std=0.02)
            if isinstance(m,nn.Linear) and m.bias is not None:nn.init.zeros_(m.bias)
    def forward(self,idx,targets=None):
        b,t=idx.shape
        assert t<=self.cfg.block_size
        x=self.tok(idx)+self.pos(torch.arange(t,device=idx.device))
        for block in self.blocks:x=block(x)
        x=self.norm(x)
        logits=self.head(x)
        loss=None if targets is None else F.cross_entropy(logits.reshape(-1,logits.size(-1)).float(),targets.reshape(-1),ignore_index=-100)
        return logits,loss
model=MaiaGPT(cfg)
params=sum(p.numel() for p in model.parameters())
print('Parameters:',format(params,','),'(~%.3f B)'%(params/1e9))
assert 480_000_000<=params<=530_000_000
assert model.head.weight.data_ptr()==model.tok.weight.data_ptr()

Parameters: 507,026,432 (~0.507 B)


In [ ]:
import torch

p = torch.cuda.get_device_properties(0)

print("GPU:", p.name)
print(f"VRAM: {p.total_memory / 2**30:.2f} GiB")
print("BF16:", torch.cuda.is_bf16_supported())

!nvidia-smi

GPU: NVIDIA A100-SXM4-40GB
VRAM: 39.49 GiB
BF16: True
Mon Nov 16 00:24:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             43W /  400W |       6MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disab

## 3 — Start or resume real training

## 3 — Continuous training (cost-minimized)

Training starts directly from the last valid checkpoint. No fixed validation, tokenizer-hash scans, corpus audit, or AdamW audit in the training run. Keep loss and gradient logging; check results manually. Periodic checkpoint interval is 10,000 **global** updates, with an additional save on controlled session end.


In [ ]:
# 3 — One training cell. Re-run this cell after any runtime restart (after cells above).
import contextlib, traceback
from collections import defaultdict
from pathlib import Path

# Explicitly chosen experimental data policy; this does NOT mark any shard as licensed.
DATA_POLICY = 'experimental_unreviewed'  # or 'reviewed_only'
# Maximum training-session wall time (excluding corpus SSD preparation).
# Choose a conservative value for your Colab allocation; 20 hours is a default, not a guarantee.
MAX_SESSION_HOURS = 20.0
STOP_RESERVE_MINUTES = 20.0  # Leave time for checkpoint I/O and session cleanup
MIN_STOP_RESERVE_SECONDS = 180.0
assert MAX_SESSION_HOURS > 0 and STOP_RESERVE_MINUTES >= 0
MICROBATCH = 2
ACCUM = 8
SEQ = 2047
LR = 3e-4
MIN_LR = 3e-5
WARMUP = 2000
WEIGHT_DECAY = 0.1
GRAD_CLIP = 1.0
CHECKPOINT_EVERY = 10000
RUN_TRAINING = True  # continuous training; restore existing checkpoint
VALIDATE_EVERY = 0  # disabled to save GPU compute
VALIDATION_BLOCKS_PER_SOURCE = 100
AUDIT_EVERY = 0  # disabled
BACKUP_EVERY = 0  # no extra 5+ GiB milestone copies
METRIC_LOG_EVERY = 20
MAX_CONSECUTIVE_LOSS_SPIKES = 20
LOSS_SPIKE_ABSOLUTE = 15.0  # conservative diagnostic guard, not a LR adjustment
TOKENIZER_DIR = Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3/tokenizer')
EXPECTED_TOKENIZER_HASHES = {
    'merges.txt': 'b4b5c08bd27684fb3c6426ffe33adaaf7375340c733dad2f1d609d90767cef8a',
    'tokenizer_config.json': 'a1cf74974f6d8a98aef2b7a2bb34084a6ad9d79c4e36e7280d369628149c7062',
    'vocab.json': '63b18c80a572b14e90de3529ac068b76ca49e449b7c4219eccbc05d6cee4cf6c',
    'tokenizer.json': 'fd6cc1c4c1a487c5c4b45ee812762c471e5776ff1845fa48ad04c245ae4644a6',
}
ALLOW_NEW_RUN = False  # Set True ONLY if confirmed no usable checkpoint remains and starting anew is intended
LOG_EVERY = 5
TARGET_TOKENS = 20 * 507_026_432
RUN_ID = 'maia_lite_0_3_0_a100_v3_experimental' if DATA_POLICY == 'experimental_unreviewed' else 'maia_lite_0_3_0_a100_v3_reviewed'
RUN_DIR = PROJECT / 'training_runs' / RUN_ID
SEED = 20261105
SOURCE_WEIGHTS = {'fineweb2_pt':.30,'wikipedia_pt':.10,'fineweb_edu_en':.25,'wikipedia_en':.10,'fineweb2_es':.20,'wikipedia_es':.05}

assert DATA_POLICY in ('experimental_unreviewed','reviewed_only')
assert MICROBATCH >= 1 and ACCUM >= 1
assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(), 'A100 CUDA BF16 required'
#assert torch.cuda.get_device_properties(0).total_memory >= 70 * 2**30, 'Select A100 80GB runtime'
gpu = torch.cuda.get_device_properties(0)
vram_gib = gpu.total_memory / 2**30

assert "A100" in gpu.name, f"GPU não suportada: {gpu.name}"
assert vram_gib >= 38, f"VRAM insuficiente: {vram_gib:.2f} GiB"
assert torch.cuda.is_bf16_supported(), "BF16 obrigatório"

print(f"GPU validada: {gpu.name} ({vram_gib:.2f} GiB)")
assert MANIFEST.is_file(), 'Run SSD preparation first'
info = json.loads(MANIFEST.read_text())
assert info['status']=='complete' and info['total_blocks']==3607119 and info['total_shards']==60612
assert info['tokens_per_block']==2048 and info['dtype']=='<u4'
entries = {(e['source'],e['split']):e for e in info['splits']}
assert len(entries)==18
for e in info['splits']:
    assert (SSD/e['relative_path']).stat().st_size == e['blocks'] * 2048 * 4
    assert (SSD/e['index_path']).is_file()

approved = {}
approval_sha = None
if DATA_POLICY == 'reviewed_only':
    assert APPROVALS.is_file(), 'Reviewed-only mode requires the original approval file'
    approval_sha = hashlib.sha256(APPROVALS.read_bytes()).hexdigest()
    for line in APPROVALS.read_text().splitlines():
        if not line.strip(): continue
        a=json.loads(line)
        key=(a['source'],a['split'],int(a['shard_index']))
        assert key not in approved
        if a.get('eligible_for_training') is True:
            assert a.get('rights_status')=='approved' and a.get('reviewer') and a.get('evidence')
            approved[key]=a

ranges=defaultdict(list)
for (source,split),e in sorted(entries.items()):
    if split!='train': continue
    end=0
    with (SSD/e['index_path']).open() as f:
        for line in f:
            r=json.loads(line); start=int(r['start_block']); count=int(r['blocks'])
            assert start==end and count>0
            end=start+count
            key=(source,split,int(r['shard_index']))
            if DATA_POLICY=='experimental_unreviewed' or key in approved:
                if key in approved: assert approved[key]['sha256'].lower()==r['sha256'].lower()
                ranges[source].append((start,count))
    assert end==e['blocks']
if DATA_POLICY=='reviewed_only':
    assert all(k[1]=='train' and k[:2] in entries for k in approved)
assert any(ranges.values()), 'No eligible training blocks'

class TrainCorpus:
    def __init__(self):
        self.ranges={}; self.ends={}; self.maps={}
        for source,rr in ranges.items():
            if not rr or SOURCE_WEIGHTS.get(source,0)<=0: continue
            e=entries[(source,'train')]
            self.maps[source]=np.memmap(SSD/e['relative_path'],dtype='<u4',mode='r',shape=(e['blocks'],2048))
            self.ranges[source]=rr
            self.ends[source]=np.cumsum([n for _,n in rr],dtype=np.int64)
        self.sources=sorted(self.maps)
        assert self.sources
        weights=np.array([SOURCE_WEIGHTS[s] for s in self.sources],dtype=np.float64)
        self.probs=weights/weights.sum()
    def batch(self,step,micro):
        digest=hashlib.sha256(f'{SEED}:{step}:{micro}'.encode()).digest()
        rng=np.random.default_rng(int.from_bytes(digest[:8],'little'))
        selected=rng.choice(len(self.sources),size=MICROBATCH,p=self.probs)
        x=np.empty((MICROBATCH,SEQ),dtype=np.int64); y=np.empty_like(x)
        for i,which in enumerate(selected):
            source=self.sources[int(which)]
            ordinal=int(rng.integers(0,int(self.ends[source][-1])))
            j=int(np.searchsorted(self.ends[source],ordinal,side='right'))
            before=0 if j==0 else int(self.ends[source][j-1])
            tokens=self.maps[source][self.ranges[source][j][0]+ordinal-before]
            assert tokens.max()<cfg.vocab_size
            x[i]=tokens[:-1];y[i]=tokens[1:]
        return torch.from_numpy(x),torch.from_numpy(y)

corpus=TrainCorpus()
TOKENS_PER_UPDATE=MICROBATCH*ACCUM*SEQ
TARGET_UPDATES=math.ceil(TARGET_TOKENS/TOKENS_PER_UPDATE)
identity={'run_id':RUN_ID,'config':asdict(cfg),'params':507026432,
          'data_policy':DATA_POLICY,'approval_sha256':approval_sha,
          'dataset_blocks':info['total_blocks'],'sources':SOURCE_WEIGHTS,
          'microbatch':MICROBATCH,'accum':ACCUM,'seq':SEQ,'seed':SEED,
          'lr':LR,'min_lr':MIN_LR,'warmup':WARMUP,'weight_decay':WEIGHT_DECAY,
          'grad_clip':GRAD_CLIP,'target_updates':TARGET_UPDATES,
          'torch_version':torch.__version__}
assert sum(p.numel() for p in model.parameters())==507026432

def lr_at(step):
    if step<WARMUP: return LR*(step+1)/WARMUP
    frac=min(1.,max(0.,(step-WARMUP)/max(1,TARGET_UPDATES-WARMUP)))
    return MIN_LR+0.5*(LR-MIN_LR)*(1+math.cos(math.pi*frac))

# Refuse accidental fallback to the ephemeral Colab filesystem.
assert Path('/content/drive/MyDrive').is_mount() or os.path.ismount('/content/drive'), 'Mount Google Drive before training'
assert PROJECT.is_dir(), f'Google Drive project directory missing: {PROJECT}'
RUN_DIR.mkdir(parents=True,exist_ok=True)
assert str(RUN_DIR.resolve()).startswith('/content/drive/MyDrive/'), 'Checkpoint destination is not Google Drive'

# Use a single stable latest file, with previous checkpoint preserved until new write completes.
LATEST=RUN_DIR/'latest.pt'
PREVIOUS=RUN_DIR/'previous.pt'
TMP=RUN_DIR/'latest.pt.tmp'

def save_checkpoint(step):
    state={'identity':identity,'step':step,'model':model.state_dict(),
           'optimizer':optimizer.state_dict(),'python_rng':random.getstate(),
           'numpy_rng':np.random.get_state(),'torch_rng':torch.get_rng_state(),
           'cuda_rng':torch.cuda.get_rng_state_all()}
    t=time.perf_counter()
    # Guard against accumulating a stale partial write after an interrupted save.
    if TMP.exists():
        print(f'WARNING: stale temporary checkpoint {TMP} ({TMP.stat().st_size/2**30:.2f} GiB); removing temporary only',flush=True)
        TMP.unlink()
    free_local=shutil.disk_usage('/content').free
    if free_local < 12*2**30:
        raise RuntimeError(f'Insufficient Colab local free space for safe checkpoint: {free_local/2**30:.2f} GiB')
    torch.save(state,TMP)
    with TMP.open('rb') as f: os.fsync(f.fileno())
    assert TMP.stat().st_size > 4*2**30, 'Checkpoint unexpectedly small; refusing promotion'
    # Validate full serialization before promoting the temporary checkpoint.
    verify=torch.load(TMP,map_location='cpu',weights_only=False)
    assert verify['identity']==identity and int(verify['step'])==step, 'Checkpoint verification failed'
    del verify; gc.collect()
    # Keep latest intact until a verified replacement exists. Copy backup first.
    if LATEST.exists():
        backup_tmp = RUN_DIR/'previous.pt.tmp'
        if backup_tmp.exists(): backup_tmp.unlink()
        shutil.copyfile(LATEST, backup_tmp)
        assert backup_tmp.stat().st_size == LATEST.stat().st_size
        # Read back backup header/identity before replacing previous.pt.
        backup_verify = torch.load(backup_tmp, map_location='cpu', weights_only=False)
        assert backup_verify['identity'] == identity and int(backup_verify['step']) < step
        del backup_verify; gc.collect()
        os.replace(backup_tmp, PREVIOUS)
    os.replace(TMP,LATEST)
    print(f'CHECKPOINT step={step} size_GiB={LATEST.stat().st_size/2**30:.2f} save_s={time.perf_counter()-t:.1f} local_free_GiB={shutil.disk_usage("/content").free/2**30:.2f}',flush=True)
    del state

def restore_checkpoint(path):
    print('RESTORE:',path,flush=True)
    ck=torch.load(path,map_location='cpu',weights_only=False)
    assert ck['identity']==identity, 'Checkpoint identity/configuration mismatch'
    model.load_state_dict(ck['model'])
    optimizer.load_state_dict(ck['optimizer'])
    # AdamW moments follow parameters to CUDA; scalar step counters may stay on CPU.
    for param,st in optimizer.state.items():
        for name in ('exp_avg','exp_avg_sq'):
            if name in st:
                assert st[name].device==param.device and st[name].shape==param.shape
    random.setstate(ck['python_rng']);np.random.set_state(ck['numpy_rng'])
    torch.set_rng_state(ck['torch_rng']);torch.cuda.set_rng_state_all(ck['cuda_rng'])
    step=int(ck['step']);del ck;gc.collect()
    return step

print('DATA POLICY:',DATA_POLICY,'(not a rights approval)',flush=True)
print('SOURCES:',corpus.sources,'TOKENS/UPDATE:',TOKENS_PER_UPDATE,'TARGET UPDATES:',TARGET_UPDATES,flush=True)
print('GPU:',torch.cuda.get_device_name(0),'RUN DIR:',RUN_DIR,flush=True)
# Recreate deterministic initialization if running this cell in a fresh runtime.
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
model=model.to('cuda')
optimizer=torch.optim.AdamW(model.parameters(),lr=LR,betas=(.9,.95),eps=1e-8,weight_decay=WEIGHT_DECAY)
start=0
if LATEST.exists() or PREVIOUS.exists():
    path=LATEST if LATEST.exists() else PREVIOUS
    try:
        start=restore_checkpoint(path)
    except Exception as exc:
        if path==LATEST and PREVIOUS.exists():
            print('WARNING: latest checkpoint failed; attempting previous checkpoint:',repr(exc),flush=True)
            start=restore_checkpoint(PREVIOUS)
        else:
            raise
    print('RESUMED AT:',start,flush=True)
    print('NEXT UPDATE:',start+1,flush=True)
else:
    print('NEW RUN: initializing training from model weights',flush=True)
    if not ALLOW_NEW_RUN:
        raise RuntimeError('No checkpoint found. Set ALLOW_NEW_RUN=True explicitly ONLY if restarting from scratch is intended.')

# --- AUDIT v3.6: read-only diagnostics (except JSONL report) ---
import itertools
AUDIT_DIR = RUN_DIR/'audit_v3_7'
AUDIT_DIR.mkdir(parents=True,exist_ok=True)
AUDIT_JSONL = AUDIT_DIR/'audit.jsonl'
TRAIN_METRICS_JSONL = AUDIT_DIR/'training_metrics.jsonl'
VALIDATION_JSONL = AUDIT_DIR/'validation_metrics.jsonl'
BACKUP_DIR = RUN_DIR/'milestone_backups'
BACKUP_DIR.mkdir(exist_ok=True)

def append_jsonl(path, record):
    with path.open('a', encoding='utf8') as f:
        f.write(json.dumps(record, ensure_ascii=False, default=str) + '\n')
        f.flush()

def verify_tokenizer_files():
    assert TOKENIZER_DIR.is_dir(), f'Tokenizer directory missing: {TOKENIZER_DIR}'
    for name, expected in EXPECTED_TOKENIZER_HASHES.items():
        p=TOKENIZER_DIR/name
        assert p.is_file(), f'Missing tokenizer artifact: {p}'
        with p.open('rb') as f:
            h=hashlib.file_digest(f,'sha256').hexdigest()
        assert h==expected, f'Tokenizer SHA256 mismatch: {name}'
    from transformers import AutoTokenizer
    tok=AutoTokenizer.from_pretrained(str(TOKENIZER_DIR), local_files_only=True, use_fast=True, trust_remote_code=False)
    assert len(tok)==cfg.vocab_size and tok.eos_token_id==2 and tok.pad_token_id==1
    print('TOKENIZER VERIFIED: four SHA256 hashes, vocab=50257, EOS=2, PAD=1',flush=True)
    append_jsonl(AUDIT_JSONL, {'utc':datetime.datetime.now(datetime.timezone.utc).isoformat(), 'kind':'tokenizer_verified', 'step':start, 'sha256':EXPECTED_TOKENIZER_HASHES})
    return tok

def archive_milestone(step):
    # Independent immutable milestone copy; does not overwrite prior archives.
    dest=BACKUP_DIR/f'checkpoint-{step:07d}.pt'
    if dest.exists():
        print('MILESTONE already present:',dest,flush=True)
        return
    temp=BACKUP_DIR/(dest.name+'.tmp')
    if temp.exists(): temp.unlink()
    shutil.copyfile(LATEST,temp)
    assert temp.stat().st_size==LATEST.stat().st_size
    ck=torch.load(temp,map_location='cpu',weights_only=False)
    assert ck['identity']==identity and int(ck['step'])==step
    del ck; gc.collect()
    os.replace(temp,dest)
    print('MILESTONE ARCHIVED:',dest,flush=True)


def emit_audit(kind, **values):
    record={'utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
            'kind':kind,'checkpoint_step':start,**values}
    append_jsonl(AUDIT_JSONL,record)
    print('AUDIT',kind,json.dumps(values,ensure_ascii=False,default=str),flush=True)

def audit_optimizer(expected_step=None):
    expected_step = start if expected_step is None else int(expected_step)
    states=list(optimizer.state.items())
    assert states, 'Checkpoint has no optimizer states'
    counts={}; finite=True; shape_ok=True
    for param,st in states:
        for name in ('exp_avg','exp_avg_sq'):
            t=st.get(name)
            shape_ok &= (t is not None and t.shape==param.shape)
            if t is not None: finite &= bool(torch.isfinite(t).all().item())
        s=st.get('step',None)
        if s is not None:
            v=int(s.item()) if torch.is_tensor(s) else int(s)
            counts[v]=counts.get(v,0)+1
    emit_audit('adamw',states=len(states),step_counts=counts,
               finite=finite,shape_ok=shape_ok,checkpoint=expected_step,
               expected_next_lr=lr_at(expected_step))
    assert finite and shape_ok and counts and set(counts)=={expected_step}, 'AdamW state inconsistent'

def audit_corpus():
    stats={}
    for source in corpus.sources:
        e=entries.get((source,'train'))
        if not e: continue
        arr=corpus.maps[source]
        count=min(16,len(arr))
        indices=np.linspace(0,len(arr)-1,num=count,dtype=np.int64)
        sample=np.asarray(arr[indices],dtype=np.int64)
        stats[source]={'blocks':len(arr),'sample_min':int(sample.min()),
                       'sample_max':int(sample.max()),
                       'sample_unique_ratio':float(len(np.unique(sample))/sample.size),
                       'sample_zero_fraction':float(np.mean(sample==0))}
        assert sample.min()>=0 and sample.max()<cfg.vocab_size
    emit_audit('corpus_token_ids',sources=stats,manifest_blocks=info['total_blocks'])
    emit_audit('corpus_sample_status',status='TOKEN_ID_RANGES_OK',note='Sample checks do not prove all blocks intact')

def fixed_validation(max_blocks_per_source=VALIDATION_BLOCKS_PER_SOURCE, step=None):
    # Use held-out val/test splits ONLY; do not reuse train samples as validation.
    model.eval()
    results={}
    with torch.inference_mode():
        for source in corpus.sources:
            key=(source,'val') if (source,'val') in entries else (source,'validation')
            if key not in entries: key=(source,'test')
            if key not in entries:
                results[source]={'status':'NO_HELDOUT_SPLIT'}
                continue
            e=entries[key]
            if int(e['blocks'])==0:
                results[source]={'status':'EMPTY_SPLIT'}
                continue
            mm=np.memmap(SSD/e['relative_path'],dtype='<u4',mode='r',shape=(e['blocks'],2048))
            rng=np.random.default_rng(int.from_bytes(hashlib.sha256(('maia-v3.6-val-'+source).encode()).digest()[:8],'little'))
            indices=rng.choice(len(mm),size=min(max_blocks_per_source,len(mm)),replace=False)
            losses=[]
            for idx in indices:
                toks=np.asarray(mm[int(idx)],dtype=np.int64)
                assert toks.min()>=0 and toks.max()<cfg.vocab_size
                x=torch.from_numpy(toks[:-1].copy()).unsqueeze(0).to('cuda')
                y=torch.from_numpy(toks[1:].copy()).unsqueeze(0).to('cuda')
                with torch.autocast('cuda',dtype=torch.bfloat16):
                    _,loss=model(x,y)
                losses.append(float(loss.item()))
            results[source]={'status':'OK','mean_loss':sum(losses)/len(losses),
                             'n_blocks':len(losses),'split':key[1]}
            del mm
    model.train()
    measured_step=start if step is None else int(step)
    record={'utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
            'step':measured_step,'blocks_per_source':max_blocks_per_source,'results':results}
    append_jsonl(VALIDATION_JSONL,record)
    print('VALIDATION',json.dumps(record,ensure_ascii=False),flush=True)
    return results

assert start>0, 'No existing checkpoint recovered; refusing automatic scratch training'
assert 0 < start <= TARGET_UPDATES
print('FAST RESUME: skipping tokenizer hash scan, corpus audit, optimizer audit, and validation',flush=True)
print('TRAINING ENABLED:',RUN_TRAINING,flush=True)

if not RUN_TRAINING:
    print('SAFE STOP: RUN_TRAINING=False',flush=True)
else:
    if start >= TARGET_UPDATES:
        print('TRAINING COMPLETE: checkpoint already at global target',start,flush=True)
    else:
        # No extra initial archive: avoids a costly multi-GB Drive copy.
        # latest.pt and previous.pt remain managed by save_checkpoint().
        model.train()
        stop=TARGET_UPDATES
        session_start=time.monotonic()
        deadline=session_start+MAX_SESSION_HOURS*3600
        reserve=max(MIN_STOP_RESERVE_SECONDS,STOP_RESERVE_MINUTES*60)
        update_seconds_ema=None
        checkpoint_seconds_ema=360.0  # initial conservative budget, adapt to measured saves
        last_completed=start
        last_saved=start
        last_validation=start
        spikes=0
        loss_ema=None
        print('TRAINING CONTINUOUS updates:',start+1,'to',stop,
              '| max_session_hours:',MAX_SESSION_HOURS,
              '| reserve_minutes:',STOP_RESERVE_MINUTES,
              '| checkpoint_every:',CHECKPOINT_EVERY,flush=True)

        def persist(step, reason):
            global last_saved, checkpoint_seconds_ema
            if step <= last_saved:
                return
            t_save=time.monotonic()
            save_checkpoint(step)
            duration=time.monotonic()-t_save
            checkpoint_seconds_ema=0.7*checkpoint_seconds_ema+0.3*duration
            last_saved=step
            append_jsonl(TRAIN_METRICS_JSONL,{
                'utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
                'kind':'checkpoint','step':step,'reason':reason,
                'duration_s':duration,'checkpoint_path':str(LATEST)})
            if BACKUP_EVERY and step % BACKUP_EVERY==0:
                archive_milestone(step)

        try:
            for step in range(start,stop):
                remaining=deadline-time.monotonic()
                predicted_next=update_seconds_ema if update_seconds_ema is not None else 120.0
                required=reserve+predicted_next+max(checkpoint_seconds_ema*1.5,300.0)
                if remaining <= required:
                    print(f'TIME BUDGET: stopping after update={last_completed}, remaining_s={remaining:.0f}',flush=True)
                    persist(last_completed,'session_time_budget')
                    break

                t0=time.perf_counter()
                optimizer.zero_grad(set_to_none=True)
                rate=lr_at(step)
                for group in optimizer.param_groups: group['lr']=rate
                loss_sum=0.0
                for micro in range(ACCUM):
                    x,y=corpus.batch(step,micro)
                    x=x.to('cuda'); y=y.to('cuda')
                    with torch.autocast('cuda',dtype=torch.bfloat16):
                        _,loss=model(x,y)
                    (loss/ACCUM).backward()
                    loss_sum+=float(loss.detach())/ACCUM
                    del x,y,loss
                norm=torch.nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP)
                grad_norm=float(norm)
                assert math.isfinite(loss_sum) and math.isfinite(grad_norm), 'Nonfinite loss/gradient; stop before optimizer update'
                # Do not silently change LR or optimizer on suspicious loss values.
                spikes = spikes+1 if loss_sum > LOSS_SPIKE_ABSOLUTE else 0
                if spikes >= MAX_CONSECUTIVE_LOSS_SPIKES:
                    raise RuntimeError(f'Persistent high training loss at update={step+1}: {spikes} consecutive updates')
                optimizer.step()
                torch.cuda.synchronize()
                elapsed=time.perf_counter()-t0
                completed=step+1
                last_completed=completed
                update_seconds_ema=elapsed if update_seconds_ema is None else 0.9*update_seconds_ema+0.1*elapsed
                loss_ema=loss_sum if loss_ema is None else 0.98*loss_ema+0.02*loss_sum

                if completed % LOG_EVERY==0:
                    print(f'update={completed}/{TARGET_UPDATES} loss={loss_sum:.5f} loss_ema={loss_ema:.5f} grad={grad_norm:.3f} lr={rate:.3e} sec={elapsed:.2f} tok/s={TOKENS_PER_UPDATE/elapsed:.0f} peak_GiB={torch.cuda.max_memory_allocated()/2**30:.2f}',flush=True)
                if completed % METRIC_LOG_EVERY==0:
                    append_jsonl(TRAIN_METRICS_JSONL,{
                        'utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
                        'kind':'train','step':completed,'loss':loss_sum,'loss_ema':loss_ema,
                        'grad_norm_preclip':grad_norm,'lr':rate,'seconds':elapsed,
                        'tokens_per_second':TOKENS_PER_UPDATE/elapsed})

                if VALIDATE_EVERY and completed % VALIDATE_EVERY==0:
                    # Fixed validation uses deterministic held-out blocks and does not update weights.
                    fixed_validation(step=completed)
                    last_validation=completed
                if completed % CHECKPOINT_EVERY==0 or completed==stop:
                    persist(completed,'periodic' if completed!=stop else 'global_target')
                if AUDIT_EVERY and completed % AUDIT_EVERY==0:
                    audit_optimizer(expected_step=completed)
                    audit_corpus()

        except BaseException as exc:
            print('TRAINING STOPPED:',type(exc).__name__,str(exc),flush=True)
            # If the optimizer update completed, try to preserve that boundary.
            # Do not suppress the original failure; never claim a save succeeded if it did not.
            if last_completed>last_saved:
                try:
                    persist(last_completed,'interruption_or_exception')
                except Exception as save_exc:
                    print('EMERGENCY CHECKPOINT FAILED:',repr(save_exc),flush=True)
            raise
        else:
            if last_completed>last_saved:
                persist(last_completed,'session_end')

        print(f'SESSION ENDED: last_completed={last_completed}, last_saved={last_saved}, target={TARGET_UPDATES}',flush=True)
        print('LOCAL FREE GiB:',round(shutil.disk_usage('/content').free/2**30,2),flush=True)
        for name in ('latest.pt','previous.pt','latest.pt.tmp'):
            p=RUN_DIR/name
            print('CHECKPOINT AUDIT',name,'exists=',p.exists(),
                  'size_GiB=',round(p.stat().st_size/2**30,2) if p.exists() else None,flush=True)


Streaming output truncated to the last 5000 lines.
update=284660/309616 loss=2.75482 loss_ema=2.83353 grad=0.966 lr=3.436e-05 sec=1.21 tok/s=26988 peak_GiB=16.29
update=284665/309616 loss=2.81960 loss_ema=2.83316 grad=1.023 lr=3.436e-05 sec=1.21 tok/s=26963 peak_GiB=16.29
update=284670/309616 loss=2.87346 loss_ema=2.83130 grad=1.037 lr=3.436e-05 sec=1.22 tok/s=26827 peak_GiB=16.29
update=284675/309616 loss=2.46574 loss_ema=2.82261 grad=0.935 lr=3.436e-05 sec=1.21 tok/s=27010 peak_GiB=16.29
update=284680/309616 loss=2.92333 loss_ema=2.82742 grad=1.051 lr=3.435e-05 sec=1.22 tok/s=26939 peak_GiB=16.29
update=284685/309616 loss=2.63188 loss_ema=2.82817 grad=0.997 lr=3.435e-05 sec=1.22 tok/s=26933 peak_GiB=16.29
update=284690/309616 loss=2.76265 loss_ema=2.82279 grad=1.007 lr=3.435e-05 sec=1.21 tok/s=26976 peak_GiB=16.29
update=284695/309616 loss=2.65046 loss_ema=2.82431 grad=1.020 lr=3.435e-05 sec=1.22 tok/s=26872 peak_GiB=16.29
update=284700/309616 loss=2.72430 loss_ema=2.82218 grad=0.978